In [1]:
!pip install gnews

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 13.3 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=76e7c1949938c3b29b43e7b0cc509e210d1abb59ecfdbef3ab7584e50c358d52
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


In [16]:
import pandas as pd, numpy as np

import re, pickle

from gnews import GNews

from sklearn.model_selection import train_test_split as tts
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression as LR
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

from sklearn.svm import SVC

In [17]:
MODEL_NAME   = "all-mpnet-base-v2"
USE_LIVE_RAG = True

In [18]:
trusted_fact_database = [
    # --- US Elections & Politics ---
    "donald trump won the 2016 presidential election with 306 electoral college votes",
    "hillary clinton won the popular vote in the 2016 presidential election by nearly three million votes",
    "the fbi investigated hillary clinton use of private email server while secretary of state",
    "fbi director james comey announced no criminal charges recommended against hillary clinton",
    "barack obama served two terms as president of the united states from 2009 to 2017",
    "the us electoral college consists of 538 electors and a majority of 270 is needed to win",
    "congress certified the 2020 presidential election results confirming joe biden victory",
    "the january 6 2021 capitol building breach interrupted the certification of electoral votes",
    "the senate intelligence committee investigated russian interference in the 2016 election",
    "robert mueller special counsel investigation examined russian interference in us elections",
    "the mueller report did not establish that the trump campaign conspired with russia",
    "us midterm elections are held every two years to elect members of congress",
    "the affordable care act also known as obamacare was signed into law in 2010",
    "supreme court justice ruth bader ginsburg died in september 2020",
    "amy coney barrett was confirmed to the supreme court in october 2020",
    "the us senate acquitted donald trump in both impeachment trials",
    "joe biden won the 2020 presidential election defeating incumbent donald trump",

    # --- Vaccines & Public Health ---
    "health officials and scientific studies confirm vaccines do not cause autism",
    "the wakefield study claiming vaccines cause autism was retracted and found fraudulent",
    "andrew wakefield lost his medical license after his discredited autism vaccine study",
    "the mmr vaccine protects against measles mumps and rubella and is considered safe",
    "the covid 19 mrna vaccines do not alter human dna according to medical experts",
    "mrna vaccines teach cells to produce a protein that triggers an immune response",
    "the world health organization declared covid 19 a global pandemic in march 2020",
    "the us food and drug administration fda authorized covid 19 vaccines for emergency use",
    "herd immunity occurs when enough of a population becomes immune to a disease",
    "vaccine ingredients are publicly listed and do not include microchips or tracking devices",
    "the cdc recommends flu vaccines annually as the influenza virus mutates each year",
    "clinical trials test vaccines for safety and efficacy before regulatory approval",
    "polio was nearly eradicated worldwide due to widespread vaccination programs",
    "natural immunity and vaccine induced immunity both help prevent infectious disease spread",
    "covid 19 vaccines were developed using spike protein technology to stimulate immunity",

    # --- Science & Space ---
    "nasa monitors near earth objects and has found no imminent asteroid threat to earth",
    "the planetary defense coordination office tracks potentially hazardous asteroids and comets",
    "climate change is driven primarily by human greenhouse gas emissions according to nasa and noaa",
    "the intergovernmental panel on climate change ipcc reports scientific consensus on global warming",
    "carbon dioxide levels in the atmosphere have risen significantly since the industrial revolution",
    "the paris agreement is an international treaty on climate change adopted in 2015",
    "nasa confirmed water ice exists on the moon in permanently shadowed craters",
    "the james webb space telescope launched in december 2021 and began science operations in 2022",
    "black holes are regions of spacetime where gravity is so strong nothing can escape",
    "the big bang theory describes the origin of the universe approximately 13 8 billion years ago",
    "spacex successfully launched and landed reusable orbital rockets reducing launch costs",
    "the international space station has been continuously inhabited since november 2000",

    # --- Economics & Finance ---
    "the us federal reserve sets interest rates to manage inflation and economic growth",
    "the 2008 financial crisis was triggered by the collapse of the subprime mortgage market",
    "the dodd frank act was passed in 2010 to regulate financial institutions after the 2008 crisis",
    "us gdp is measured quarterly and represents the total economic output of the country",
    "the unemployment rate measures the percentage of the labor force actively seeking work",
    "tariffs are taxes imposed on imported goods and can raise prices for consumers",
    "the stock market experienced significant volatility during the covid 19 pandemic in 2020",
    "bitcoin is a decentralized digital currency not backed by any government or central bank",

    # --- Law Enforcement & Legal ---
    "the department of justice oversees federal law enforcement agencies including the fbi",
    "the fourth amendment protects americans from unreasonable searches and seizures",
    "the first amendment protects freedom of speech press religion and assembly",
    "the second amendment protects the right to keep and bear arms",
    "the supreme court ruled in citizens united that political spending is protected speech",
    "plea bargains resolve the majority of criminal cases in the us court system",
    "the patriot act expanded surveillance powers of us intelligence agencies after september 11",
    "edward snowden leaked classified nsa documents revealing mass surveillance programs in 2013",

    # --- Media & Misinformation ---
    "facebook and twitter implemented fact checking labels on posts containing misinformation",
    "the term fake news refers to deliberate disinformation presented as legitimate journalism",
    "media literacy education helps people identify credible sources and recognize bias",
    "social media algorithms can amplify misinformation due to high engagement on emotional content",
    "the associated press reuters and bbc are considered internationally recognized news sources",
    "satirical news websites like the onion publish fictional stories not intended as real news",
    "the fairness doctrine required broadcast media to present contrasting views on controversial issues",
    "deepfake technology uses artificial intelligence to create realistic fake videos of real people",

    # --- Immigration ---
    "daca deferred action for childhood arrivals protects undocumented immigrants brought as children",
    "the us border patrol is responsible for securing us borders between ports of entry",
    "immigration courts are part of the department of justice not the federal judiciary",
    "asylum seekers must demonstrate fear of persecution in their home country to qualify",
    "the immigration and nationality act establishes the legal framework for us immigration policy",

    # --- Terrorism & National Security ---
    "the september 11 2001 attacks were carried out by al qaeda killing nearly 3000 people",
    "the us invaded afghanistan in 2001 following the september 11 attacks",
    "osama bin laden was killed by us navy seals in pakistan in may 2011",
    "isis also known as isil or daesh is a jihadist militant group that emerged in iraq and syria",
    "the department of homeland security was created after september 11 to coordinate domestic security",
    "the patriot act gave law enforcement broader surveillance authority following the 9 11 attacks",

    # Add these to trusted_facts list
    "nasa artemis program aims to return humans to the moon including the first woman",
    "the space launch system sls is nasa primary rocket for deep space exploration",
    "the orion capsule is designed to carry astronauts to the moon and deep space",
    "nasa is working with esa jaxa and csa on the lunar gateway space station",
    "nasa completed an uncrewed artemis test flight gathering critical engineering data",
    "the lunar gateway is a planned space station orbiting the moon for future missions",
    "nasa plans to establish a sustainable human presence on the moon before going to mars",
    "spacex boeing and other private companies are partners in nasa commercial crew program",
  ]

In [19]:
embedder = SentenceTransformer(MODEL_NAME)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [20]:
gnews_client = GNews(
    language='en',
    country='us',
    period='30d',
    max_results=6,
)

In [21]:
def preprocessing(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [22]:
def get_embeddings(texts, batch_size=64, show_progress=False):
  if isinstance(texts, str):
    texts = [texts]
  return embedder.encode(
      texts,
      batch_size=batch_size,
      show_progress_bar=show_progress,
      convert_to_numpy=True,
  )

In [23]:
def extract_claims(text, n=5):
    sentences = [s.strip() for s in text.split('.') if len(s.strip()) > 40]
    return sentences[:n]

In [24]:
def search_gnews(query):
  try:
    results = gnews_client.get_news(query)
    snippets=[]

    for article in results:
      title = article.get('title', '')
      description = article.get('desc', '')
      combined = f"{title} {description}"
      if combined and len(combined) > 20:
        snippets.append(combined)
    return snippets
  except Exception as e:
    print(f"  GNews search failed for '{query[:40]}': {e}")
    return []

In [25]:
def fetch_live_evidence(article_text):
  claims = extract_claims(article_text)
  evidence = []
  for claim in claims:
    query = claim[:60]
    results = re.search_gnews(query)
    evidence.extend(results)

  results = list(dict.fromkeys(evidence))
  return evidence, claims

In [26]:
def compute_rag_features(article_embeddings, reference_embeddings):
  sims = cosine_similarity(article_embeddings, reference_embeddings)
  max_sim = sims.max(axis=1, keepdims=True)
  mean_sim = sims.mean(axis=1, keepdims=True)
  top3_sim = np.sort(sims, axis=1)[:, -3:].mean(axis=1, keepdims=True)
  return np.hstack([max_sim, mean_sim.reshape(-1, 1), top3_sim.reshape(-1, 1)])

In [28]:
def train():
  df = pd.read_csv('fake_or_real_news.csv', engine='python', on_bad_lines ='skip')

  le = LabelEncoder()
  df['label'] = le.fit_transform(df['label'])

  df['content'] = (df['title'].fillna('')+' '+df['text'].fillna('')).apply(preprocessing)
  df = df[df['content'].str.len() > 100].reset_index(drop=True)

  X_bert = get_embeddings(df['content'].tolist(), show_progress=True)

  fact_embeddings = get_embeddings(trusted_fact_database)

  X_rag = compute_rag_features(X_bert, fact_embeddings)
  X_combined = np.hstack([X_bert, X_rag])
  y = df['label'].values

  X_train, X_test, y_train, y_test = tts(X_combined, y, test_size=0.2, random_state=0, stratify=y)

  model = SVC(kernel='rbf', C=10, probability=True)
  model.fit(X_train, y_train)

  y_pred = model.predict(X_test)
  print(f"\nAccuracy : {accuracy_score(y_test, y_pred)*100:.2f}%")
  print(classification_report(y_test, y_pred, target_names=le.classes_))

  pickle.dump(model, open('model.pkl', 'wb'))
  pickle.dump(le, open('label_encoder.pkl', 'wb'))
  pickle.dump(fact_embeddings, open('fact_embeddings.pkl', 'wb'))
  pickle.dump(trusted_fact_database, open('trusted_facts.pkl', 'wb'))

  return model, le, trusted_fact_database

In [29]:
if __name__ == "__main__":
  clf, le, fact_embeddings = train()


Batches:   0%|          | 0/98 [00:00<?, ?it/s]


Accuracy : 92.43%
              precision    recall  f1-score   support

        FAKE       0.92      0.92      0.92       624
        REAL       0.92      0.93      0.92       631

    accuracy                           0.92      1255
   macro avg       0.92      0.92      0.92      1255
weighted avg       0.92      0.92      0.92      1255

